# FirstAI + Orbit — integrated engineering foundation\nRun the cells from top to bottom. This notebook verifies the custom GPT foundation, runs the GPU training pipeline, and includes tests for Orbit's identity, task recovery, persistent memory, and tool authorization. The baseline checkpoint is still an experimental language model, not a capable assistant.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path('/content/FirstAI')
if REPO.is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/llakshyaa-rgb/FirstAI.git', str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi is not available in this runtime.')

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GiB')
else:
    print('\nGPU CHECK NOT PASSED.')
    print('In Colab choose Runtime → Change runtime type → T4 GPU, save,')
    print('then Runtime → Disconnect and delete runtime, reconnect, and rerun from the top.')
    print('If nvidia-smi shows a GPU but PyTorch CUDA build is None, this runtime has CPU-only PyTorch.')
    print('Do not run Test 2 until CUDA available is True.')

# Safe paths are initialized before Drive authentication is attempted.
LOCAL_BASE = Path('/content/FirstAI_workspace')
BASE = LOCAL_BASE
RAW = BASE / 'data' / 'raw'
OUT = BASE / 'checkpoints'
CORPUS = str(BASE / 'data' / 'input.txt')
CHECKPOINT = str(OUT / 'firstgpt.pt')
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('\nInitial workspace:', BASE)

## Test 1 — automated FirstAI and Orbit unit tests\nChecks tokenizer, corpus preparation, data batching, model shape, causal attention, gradients, Orbit identity, task fallback/recovery, memory persistence, and tool authorization.

In [ ]:
!python -m unittest discover -s tests -v

## Test 2 — training engine and checkpoint recovery
Runs a tiny temporary training job, saves latest/best checkpoints, resumes, restores scaler/RNG state, evaluates and generates. This is an engineering test, not meaningful knowledge training.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        'Test 2 needs a CUDA-enabled Colab runtime. The current PyTorch build is '
        f'{torch.__version__} (CUDA build: {torch.version.cuda}). '
        'Select T4 GPU, reconnect the runtime, and rerun the setup cell.'
    )
!python scripts/smoke_test.py --require-cuda

## Test 3 — integrated GPU pipeline with a generated test corpus
No Google Drive, uploads or external dataset are needed. This test creates a temporary synthetic text corpus inside Colab, prepares it, trains from random initialization, resumes, evaluates and generates. It verifies the engineering pipeline only; synthetic text does not teach useful real-world knowledge.

In [ ]:
# Use only temporary Colab storage. No Drive mount or upload.
BASE = LOCAL_BASE
RAW = BASE / 'data' / 'raw'
OUT = BASE / 'checkpoints'
CORPUS = str(BASE / 'data' / 'input.txt')
CHECKPOINT = str(OUT / 'firstgpt.pt')
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Generate a deterministic, varied corpus for pipeline verification.
import random
rng = random.Random(20261113)
subjects = ["the small machine", "a curious learner", "the blue engine", "the quiet forest", "a research team", "the little robot", "an observant scientist", "the river system"]
verbs = ["studies", "builds", "measures", "describes", "tests", "discovers", "compares", "records"]
objects = ["patterns in data", "the movement of light", "how memory works", "a sequence of signals", "the structure of language", "changes in temperature", "the flow of information", "a new experiment"]
connectors = ["because", "while", "although", "when", "after", "before", "therefore", "for example"]
details = ["carefully and repeatedly", "using simple observations", "with a clear procedure", "over several experiments", "by checking each result", "without assuming the answer", "and stores the result", "then explains the outcome"]
paragraphs = []
for i in range(1800):
    subject, verb, obj = rng.choice(subjects), rng.choice(verbs), rng.choice(objects)
    connector, detail = rng.choice(connectors), rng.choice(details)
    paragraphs.append(
        f"Example {i + 1}. {subject.capitalize()} {verb} {obj} {connector} "
        f"the process can be examined {detail}. The observation number is {i + 1}, "
        f"and the measured value is {rng.randint(1, 1000)}. "
        "A useful model learns relationships between tokens from many examples."
    )
synthetic_path = RAW / 'synthetic_test_corpus.txt'
synthetic_path.write_text("\n\n".join(paragraphs), encoding='utf-8')
print('Synthetic corpus:', synthetic_path)
print('Paragraphs:', len(paragraphs), '| Characters:', sum(map(len, paragraphs)))
print('Workspace:', BASE, '| Checkpoints:', OUT)

### Prepare the generated corpus
The text file is generated locally in Colab; no file picker is needed.

In [ ]:
if not any(p.is_file() and p.suffix.lower() == '.txt' for p in RAW.rglob('*')):
    raise RuntimeError(f'No .txt corpus files found under {RAW}. Run the upload cell and select at least one .txt file.')
(BASE / 'data').mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'prepare_corpus.py',
    '--input-dir', str(RAW), '--output', CORPUS
], check=True)
if not Path(CORPUS).is_file():
    raise RuntimeError(f'Corpus preparation did not create {CORPUS}')
print('Corpus ready:', CORPUS, '| bytes:', Path(CORPUS).stat().st_size)

### Train from random initialization
Starter configuration for a T4. This is a small baseline, not a claim that the model will become a capable assistant. Training can be resumed using the next cell.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Real T4 training is blocked because CUDA is unavailable. Fix the runtime before training.')
if not Path(CORPUS).is_file():
    raise RuntimeError(f'Corpus file not found: {CORPUS}. Run the generated-corpus preparation cell first.')
subprocess.run([
    sys.executable, 'train.py', '--data', CORPUS, '--out', CHECKPOINT,
    '--steps', '1000', '--eval-interval', '100', '--checkpoint-interval', '250',
    '--batch-size', '16', '--block-size', '128', '--layers', '4', '--heads', '4',
    '--width', '128', '--grad-accum-steps', '2', '--amp'
], check=True)

### Resume training if needed
Run this cell only after a previous training run has created the checkpoint. It uses the same corpus and restores the saved model, optimizer, mixed-precision scaler and RNG states.

In [ ]:
if not Path(CHECKPOINT).is_file():
    raise RuntimeError(f'No resumable checkpoint found at {CHECKPOINT}. Run the initial training cell first.')
subprocess.run([
    sys.executable, 'train.py', '--data', CORPUS, '--out', CHECKPOINT,
    '--resume', '--steps', '1000', '--eval-interval', '100',
    '--checkpoint-interval', '250', '--batch-size', '16',
    '--grad-accum-steps', '2', '--amp'
], check=True)

### Evaluate and generate
Evaluation checks the exact corpus fingerprint. Generation uses the best validation checkpoint.

In [ ]:
BEST_CHECKPOINT = str(OUT / 'firstgpt.best.pt')
if not Path(BEST_CHECKPOINT).is_file():
    raise RuntimeError(f'Best checkpoint not found: {BEST_CHECKPOINT}. Training must finish successfully first.')
subprocess.run([
    sys.executable, 'evaluate.py', '--checkpoint', BEST_CHECKPOINT,
    '--data', CORPUS, '--batches', '50', '--batch-size', '16'
], check=True)
subprocess.run([
    sys.executable, 'generate.py', '--checkpoint', BEST_CHECKPOINT,
    '--prompt', 'The ', '--tokens', '200', '--temperature', '0.8', '--top-k', '40'
], check=True)

### Download a copy of the checkpoints
Download both files to your phone before disconnecting. Temporary Colab storage is erased when the runtime disconnects.

In [ ]:
from google.colab import files
if not Path(CHECKPOINT).is_file() or not Path(BEST_CHECKPOINT).is_file():
    raise RuntimeError('Both latest and best checkpoints must exist before downloading.')
files.download(CHECKPOINT)
files.download(BEST_CHECKPOINT)